In [1]:
# Colab için çakışma-minimum kurulum (numpy/fsspec pin yok)
!pip -q install -U   "transformers==4.44.2"   "datasets==2.21.0"   "evaluate==0.4.2"   "sacrebleu==2.4.3"   "sentencepiece==0.2.0"   "accelerate==0.34.2"   "rouge-score==0.1.2"



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 157.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 133.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the pack

In [2]:
import os
import numpy as np
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (
    MBart50TokenizerFast,
    MBartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


In [3]:
# mBART-50 model/tokenizer
MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
model = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

In [4]:
# XLSum dili -> mBART lang code eşleşmesi (seçili diller)
XLSUM_TO_MBAR = {
    "arabic": "ar_AR",
    "chinese_simplified": "zh_CN",
    "english": "en_XX",
    "french": "fr_XX",
    "hindi": "hi_IN",
    "japanese": "ja_XX",
    "korean": "ko_KR",
    "persian": "fa_IR",
    "russian": "ru_RU",
    "spanish": "es_XX",
    "turkish": "tr_TR",
    "ukrainian": "uk_UA",
    "vietnamese": "vi_VN",
    "portuguese": "pt_XX",
    "indonesian": "id_ID",
}

# tokenizer'ın gerçekten desteklediği kodlara göre son filtre
supported_codes = set(tokenizer.lang_code_to_id.keys())
XLSUM_TO_MBAR = {k: v for k, v in XLSUM_TO_MBAR.items() if v in supported_codes}

print(f"Kullanılacak dil sayısı: {len(XLSUM_TO_MBAR)}")
print(sorted(XLSUM_TO_MBAR.items()))



Kullanılacak dil sayısı: 15
[('arabic', 'ar_AR'), ('chinese_simplified', 'zh_CN'), ('english', 'en_XX'), ('french', 'fr_XX'), ('hindi', 'hi_IN'), ('indonesian', 'id_ID'), ('japanese', 'ja_XX'), ('korean', 'ko_KR'), ('persian', 'fa_IR'), ('portuguese', 'pt_XX'), ('russian', 'ru_RU'), ('spanish', 'es_XX'), ('turkish', 'tr_TR'), ('ukrainian', 'uk_UA'), ('vietnamese', 'vi_VN')]


In [5]:
# Eğitim hızını Colab'a göre ayarlamak için opsiyonlar
MAX_TRAIN_PER_LANG = None
MAX_VAL_PER_LANG = 200
MAX_TEST_PER_LANG = 200

SOURCE_MAX_LEN = 512
TARGET_MAX_LEN = 96

from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/mbart50-xlsum-multilingual"
print(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)


Mounted at /content/drive
/content/drive/MyDrive/mbart50-xlsum-multilingual


In [6]:
# Tüm dilleri oku, sample et, birleştir
train_parts, val_parts, test_parts = [], [], []

for xlsum_lang, mbart_code in XLSUM_TO_MBAR.items():
    ds = load_dataset("GEM/xlsum", xlsum_lang)

    tr = ds["train"]
    va = ds["validation"]
    te = ds["test"]

    # Bazı mirror/sürümlerde hedef kolon adı summary olabiliyor
    for split_name, split_ds in [("train", tr), ("validation", va), ("test", te)]:
        if "target" not in split_ds.column_names and "summary" in split_ds.column_names:
            if split_name == "train":
                tr = split_ds.rename_column("summary", "target")
            elif split_name == "validation":
                va = split_ds.rename_column("summary", "target")
            else:
                te = split_ds.rename_column("summary", "target")

    if MAX_TRAIN_PER_LANG is not None:
        tr = tr.shuffle(seed=SEED).select(range(min(MAX_TRAIN_PER_LANG, len(tr))))
    if MAX_VAL_PER_LANG is not None:
        va = va.shuffle(seed=SEED).select(range(min(MAX_VAL_PER_LANG, len(va))))
    if MAX_TEST_PER_LANG is not None:
        te = te.shuffle(seed=SEED).select(range(min(MAX_TEST_PER_LANG, len(te))))

    tr = tr.add_column("mbart_lang", [mbart_code] * len(tr))
    va = va.add_column("mbart_lang", [mbart_code] * len(va))
    te = te.add_column("mbart_lang", [mbart_code] * len(te))

    train_parts.append(tr)
    val_parts.append(va)
    test_parts.append(te)

train_ds = concatenate_datasets(train_parts).shuffle(seed=SEED)
val_ds = concatenate_datasets(val_parts).shuffle(seed=SEED)
test_ds = concatenate_datasets(test_parts).shuffle(seed=SEED)

print("Train:", len(train_ds), "Validation:", len(val_ds), "Test:", len(test_ds))


The repository for GEM/xlsum contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/GEM/xlsum.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Flattening the indices:   0%|          | 0/200 [00:00<?, ? examples/s]

...

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Flattening the indices:   0%|          | 0/200 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/200 [00:00<?, ? examples/s]

Train: 818129 Validation: 3000 Test: 3000


In [7]:
# Dil bazlı tokenization (her örnekte source/target language kodu set edilir)
def preprocess(batch):
    input_ids, attention_masks, labels = [], [], []

    for text, summary, lang_code in zip(batch["text"], batch["target"], batch["mbart_lang"]):
        tokenizer.src_lang = lang_code
        tokenizer.tgt_lang = lang_code

        enc = tokenizer(
            text,
            max_length=SOURCE_MAX_LEN,
            truncation=True,
            padding=False,
        )

        dec = tokenizer(
            text_target=summary,
            max_length=TARGET_MAX_LEN,
            truncation=True,
            padding=False,
        )

        input_ids.append(enc["input_ids"])
        attention_masks.append(enc["attention_mask"])
        labels.append(dec["input_ids"])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        "labels": labels,
    }

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)
test_tok = test_ds.map(preprocess, batched=True, remove_columns=test_ds.column_names)


Map:   0%|          | 0/818129 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [8]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )

    return {k: round(v * 100, 4) for k, v in result.items()}


In [31]:
# Colab T4/L4 için güvenli başlangıç ayarları
args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy="no",
    save_steps=10000,
    logging_steps=100,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=3e-5,
    num_train_epochs=2,
    warmup_ratio=0.03,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=False,
    generation_max_length=TARGET_MAX_LEN,
    bf16=torch.cuda.is_available(),
    fp16=False,
    dataloader_num_workers=2,
    load_best_model_at_end=False,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [32]:
train_result = trainer.train()
print(train_result)

save_dir = f"{OUTPUT_DIR}/final"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print("Model kaydedildi:", save_dir)

metrics = trainer.evaluate(test_tok, metric_key_prefix="test")
print(metrics)


Step,Training Loss
100,2.339200
200,2.177000
300,2.135700
400,2.103600
500,2.077800
600,2.055000
700,2.338900
800,2.438700
900,2.414900
1000,2.409800


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}


TrainOutput(global_step=6390, training_loss=2.2078418594384233, metrics={'train_runtime': 17019.0855, 'train_samples_per_second': 96.143, 'train_steps_per_second': 0.375, 'total_flos': 1.7725225263169536e+18, 'train_loss': 2.2078418594384233, 'epoch': 1.99945241913404})
Model kaydedildi: /content/drive/MyDrive/mbart50-xlsum-multilingual/final


OutOfMemoryError: CUDA out of memory. Tried to allocate 26.29 GiB. GPU 0 has a total capacity of 79.25 GiB of which 26.16 GiB is free. Including non-PyTorch memory, this process has 53.07 GiB memory in use. Of the allocated memory 38.20 GiB is allocated by PyTorch, and 14.37 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)